In [2]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('ecommerce.db')
cursor = conn.cursor()

cursor.execute("DROP TABLE IF EXISTS order_items;")
cursor.execute("DROP TABLE IF EXISTS orders;")
cursor.execute("DROP TABLE IF EXISTS products;")
cursor.execute("DROP TABLE IF EXISTS customers;")

cursor.execute("""
CREATE TABLE customers (
    customer_id INT PRIMARY KEY,
    first_name VARCHAR(50) NOT NULL,
    last_name VARCHAR(50) NOT NULL,
    email VARCHAR(100) UNIQUE NOT NULL,
    city VARCHAR(50) NOT NULL,
    state VARCHAR(50) NOT NULL,
    join_date DATE NOT NULL,
    is_premium BOOLEAN DEFAULT FALSE
);
""")

cursor.execute("""
CREATE TABLE products (
    product_id INT PRIMARY KEY,
    product_name VARCHAR(100) NOT NULL,
    category VARCHAR(50) NOT NULL,
    brand VARCHAR(50) NOT NULL,
    unit_price DECIMAL(10,2) NOT NULL CHECK (unit_price > 0),
    stock_qty INT NOT NULL DEFAULT 0 CHECK (stock_qty >= 0)
);
""")

cursor.execute("""
CREATE TABLE orders (
    order_id INT PRIMARY KEY,
    customer_id INT NOT NULL,
    order_date DATE NOT NULL,
    status VARCHAR(20) NOT NULL DEFAULT 'Pending' CHECK (status IN ('Pending', 'Shipped', 'Delivered', 'Cancelled')),
    total_amount DECIMAL(12,2) NOT NULL CHECK (total_amount >= 0),
    FOREIGN KEY (customer_id) REFERENCES customers (customer_id)
);
""")

cursor.execute("""
CREATE TABLE order_items (
    item_id INT PRIMARY KEY,
    order_id INT NOT NULL,
    product_id INT NOT NULL,
    quantity INT NOT NULL CHECK (quantity > 0),
    unit_price DECIMAL(10,2) NOT NULL CHECK (unit_price > 0),
    discount_pct DECIMAL(5,2) DEFAULT 0 CHECK (discount_pct BETWEEN 0 AND 100),
    FOREIGN KEY (order_id) REFERENCES orders (order_id),
    FOREIGN KEY (product_id) REFERENCES products (product_id)
);
""")

cursor.execute("""
INSERT INTO customers VALUES
(101, 'Aarav', 'Sharma', 'aarav.s@email.com', 'Mumbai', 'Maharashtra', '2024-01-15', 1),
(102, 'Priya', 'Patel', 'priya.p@email.com', 'Ahmedabad', 'Gujarat', '2024-02-20', 0),
(103, 'Rohan', 'Gupta', 'rohan.g@email.com', 'Delhi', 'Delhi', '2024-03-10', 1),
(104, 'Sneha', 'Reddy', 'sneha.r@email.com', 'Hyderabad', 'Telangana', '2024-04-05', 0),
(105, 'Vikram', 'Singh', 'vikram.s@email.com', 'Jaipur', 'Rajasthan', '2024-05-12', 1),
(106, 'Ananya', 'Iyer', 'ananya.i@email.com', 'Chennai', 'Tamil Nadu', '2024-06-18', 0),
(107, 'Karan', 'Mehta', 'karan.m@email.com', 'Pune', 'Maharashtra', '2024-07-22', 1),
(108, 'Divya', 'Nair', 'divya.n@email.com', 'Kochi', 'Kerala', '2024-08-30', 0);
""")

cursor.execute("""
INSERT INTO products VALUES
(201, 'Wireless Earbuds', 'Electronics', 'BoAt', 1499.00, 250),
(202, 'Cotton T-Shirt', 'Clothing', 'Levi''s', 799.00, 500),
(203, 'Smart Watch', 'Electronics', 'Noise', 2999.00, 150),
(204, 'Running Shoes', 'Clothing', 'Nike', 4599.00, 120),
(205, 'Bluetooth Speaker', 'Electronics', 'JBL', 3499.00, 200),
(206, 'Bedsheet Set', 'Home', 'Spaces', 1299.00, 300),
(207, 'Laptop Stand', 'Electronics', 'AmazonBasics', 899.00, 180),
(208, 'Cushion Covers (Set)', 'Home', 'HomeCenter', 599.00, 400);
""")

cursor.execute("""
INSERT INTO orders VALUES
(1001, 101, '2024-08-01', 'Delivered', 4498.00),
(1002, 102, '2024-08-03', 'Delivered', 799.00),
(1003, 103, '2024-08-05', 'Shipped', 7498.00),
(1004, 101, '2024-08-10', 'Delivered', 3499.00),
(1005, 104, '2024-08-12', 'Cancelled', 2999.00),
(1006, 105, '2024-08-15', 'Delivered', 5898.00),
(1007, 106, '2024-08-18', 'Pending', 1299.00),
(1008, 103, '2024-08-20', 'Delivered', 899.00),
(1009, 107, '2024-08-25', 'Shipped', 6098.00),
(1010, 108, '2024-08-28', 'Delivered', 1598.00);
""")

cursor.execute("""
INSERT INTO order_items VALUES
(5001, 1001, 201, 2, 1499.00, 0),
(5002, 1001, 207, 1, 899.00, 10),
(5003, 1002, 202, 1, 799.00, 0),
(5004, 1003, 203, 1, 2999.00, 0),
(5005, 1003, 204, 1, 4599.00, 5),
(5006, 1004, 205, 1, 3499.00, 0),
(5007, 1005, 203, 1, 2999.00, 0),
(5008, 1006, 201, 1, 1499.00, 10),
(5009, 1006, 204, 1, 4599.00, 5),
(5010, 1007, 206, 1, 1299.00, 0),
(5011, 1008, 207, 1, 899.00, 0),
(5012, 1009, 205, 1, 3499.00, 0),
(5013, 1009, 208, 2, 599.00, 15),
(5014, 1010, 206, 1, 1299.00, 0),
(5015, 1010, 208, 1, 599.00, 0);
""")
conn.commit()

def run_assignment_query(title, query_string):
    print(f"\n================ {title} ================")
    try:
        result = pd.read_sql_query(query_string, conn)
        print(result.to_string(index=False))
    except Exception as err:
        print(f"Error: {err}")

run_assignment_query("Q1. Display all customers", "SELECT * FROM customers;")
run_assignment_query("Q2. Retrieve specific customer columns", "SELECT first_name, last_name, city FROM customers;")
run_assignment_query("Q3. Unique product categories", "SELECT DISTINCT category FROM products;")

print("\n================ Q4. Primary Keys Explanation ================")
print("Primary Keys:")
print("customers: customer_id")
print("products: product_id")
print("orders: order_id")
print("order_items: item_id")
print("Explanation: A Primary Key must be unique to prevent row confusion, and NOT NULL because an empty value cannot identify a specific row.")

print("\n================ Q5. Email Constraints Explanation ================")
print("Constraints: UNIQUE and NOT NULL.")
print("If we try to insert a duplicate email, the database will raise an Integrity Constraint Error and block it.")

print("\n================ Q6. Check Constraint Verification ================")
try:
    cursor.execute("INSERT INTO products VALUES (999, 'Test Product', 'Home', 'BrandX', -50.00, 10);")
except Exception as e:
    print(f"Error caught: {e}")

run_assignment_query("Q7. Orders with status Delivered", "SELECT * FROM orders WHERE status = 'Delivered';")
run_assignment_query("Q8. Electronics priced over 2000", "SELECT * FROM products WHERE category = 'Electronics' AND unit_price > 2000;")
run_assignment_query("Q9. Customers from Maharashtra who joined in 2024", "SELECT * FROM customers WHERE state = 'Maharashtra' AND join_date LIKE '2024%';")
run_assignment_query("Q10. Non-cancelled orders between 2024-08-10 and 2024-08-25", "SELECT * FROM orders WHERE order_date BETWEEN '2024-08-10' AND '2024-08-25' AND status != 'Cancelled';")

print("\n================ Q11. Index Performance ================")
print("The index idx_orders_date pre-sorts the orders by date. Instead of scanning every row sequentially, the engine jumps directly to matching dates.")

print("\n================ Q12. SARGable Query Rewrite ================")
print("Using YEAR(join_date) = 2024 prevents index use because it recalculates values for every row.")
print("Index-friendly rewrite sample: SELECT * FROM customers WHERE join_date BETWEEN '2024-01-01' AND '2024-12-31';")

run_assignment_query("Q13. Total number of orders", "SELECT COUNT(*) AS total_orders FROM orders;")
run_assignment_query("Q14. Total revenue from Delivered orders", "SELECT SUM(total_amount) AS total_revenue FROM orders WHERE status = 'Delivered';")
run_assignment_query("Q15. Average unit price per category", "SELECT category, AVG(unit_price) AS average_price FROM products GROUP BY category;")
run_assignment_query("Q16. Order count and revenue by status", "SELECT status, COUNT(*) AS order_count, SUM(total_amount) AS total_revenue FROM orders GROUP BY status ORDER BY total_revenue DESC;")
run_assignment_query("Q17. Max and Min product prices per category", "SELECT category, MAX(unit_price) AS max_price, MIN(unit_price) AS min_price FROM products GROUP BY category;")
run_assignment_query("Q18. Categories with average price > 2000", "SELECT category, AVG(unit_price) AS average_price FROM products GROUP BY category HAVING AVG(unit_price) > 2000;")

run_assignment_query("Q19. Orders with Customer Names (INNER JOIN)",
                     "SELECT o.order_id, o.order_date, c.first_name, c.last_name, o.total_amount FROM orders o INNER JOIN customers c ON o.customer_id = c.customer_id;")

run_assignment_query("Q20. All Customers and Orders (LEFT JOIN)",
                     "SELECT c.customer_id, c.first_name, o.order_id, o.status FROM customers c LEFT JOIN orders o ON c.customer_id = o.customer_id;")

run_assignment_query("Q21. Three-table Join Details",
                     "SELECT oi.order_id, p.product_name, oi.quantity, oi.unit_price, oi.discount_pct FROM order_items oi JOIN orders o ON oi.order_id = o.order_id JOIN products p ON oi.product_id = p.product_id;")

print("\n================ Q22. Join Differences ================")
print("LEFT JOIN: Keeps all rows from left table and matches right entries.")
print("RIGHT JOIN: Keeps all rows from right table and matches left entries.")
print("FULL OUTER JOIN: Combines both, keeping all unmatched items from both tables.")

print("\n================ Q23. Foreign Key Violated Insert ================")
try:
    cursor.execute("INSERT INTO orders VALUES (2222, 999, '2024-08-15', 'Pending', 100.00);")
except Exception as e:
    print(f"Error caught: {e}")

run_assignment_query("Q24. Price Tiers (CASE Statement)",
                     "SELECT product_name, unit_price, CASE WHEN unit_price < 1000 THEN 'Budget' WHEN unit_price BETWEEN 1000 AND 3000 THEN 'Mid-Range' ELSE 'Premium' END AS price_tier FROM products;")

run_assignment_query("Q25. Conditional Aggregate Status Summary",
                     "SELECT SUM(CASE WHEN status = 'Delivered' THEN 1 ELSE 0 END) AS Delivered_Count, SUM(CASE WHEN status != 'Delivered' THEN 1 ELSE 0 END) AS Not_Delivered_Count FROM orders;")

print("\n================ Q26. ACID Properties ================")
print("Atomicity: All parts of a transaction succeed or all fail.")
print("Consistency: Transaction moves database from one valid state to another.")
print("Isolation: Concurrent transactions do not interfere with each other.")
print("Durability: Once saved, updates survive system crashes.")

print("\n================ Q27. Transaction Control Example ================")
try:
    cursor.execute("BEGIN TRANSACTION;")
    cursor.execute("INSERT INTO orders VALUES (1011, 102, '2024-08-31', 'Pending', 1598.00);")
    cursor.execute("INSERT INTO order_items VALUES (5016, 1011, 206, 1, 1299.00, 0);")
    cursor.execute("INSERT INTO order_items VALUES (5017, 1011, 208, 1, 599.00, 0);")
    cursor.execute("UPDATE products SET stock_qty = stock_qty - 1 WHERE product_id IN (206, 208);")
    cursor.execute("COMMIT;")
    print("Transaction complete and COMMITTED successfully.")
except Exception as e:
    cursor.execute("ROLLBACK;")
    print(f"Transaction aborted and ROLLED BACK due to error: {e}")

conn.close()


================ Q1. Display all customers ================
 customer_id first_name last_name              email      city       state  join_date  is_premium
         101      Aarav    Sharma  aarav.s@email.com    Mumbai Maharashtra 2024-01-15           1
         102      Priya     Patel  priya.p@email.com Ahmedabad     Gujarat 2024-02-20           0
         103      Rohan     Gupta  rohan.g@email.com     Delhi       Delhi 2024-03-10           1
         104      Sneha     Reddy  sneha.r@email.com Hyderabad   Telangana 2024-04-05           0
         105     Vikram     Singh vikram.s@email.com    Jaipur   Rajasthan 2024-05-12           1
         106     Ananya      Iyer ananya.i@email.com   Chennai  Tamil Nadu 2024-06-18           0
         107      Karan     Mehta  karan.m@email.com      Pune Maharashtra 2024-07-22           1
         108      Divya      Nair  divya.n@email.com     Kochi      Kerala 2024-08-30           0

================ Q2. Retrieve specific customer columns 